In [7]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv()

True

In [8]:
model = ChatMistralAI(model="mistral-small-2506")

In [9]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [10]:
def generate_joke(state: JokeState) -> JokeState:
    prompt = f"based on the topic create a very funny small joke topic: {state['topic']}"
    result = model.invoke(prompt).content
    return {"joke" : result}

def generate_explanation(state: JokeState) -> JokeState:
    prompt = f"based on this joke generate a small explanation maximum of 30 words"
    result = model.invoke(prompt).content
    return {"explanation" : result}

In [11]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_joke", END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

config1 = {"configurable" : {"thread_id" : "1"}}
result = workflow.invoke({
    "topic": "pizza"
}, config=config1)


In [12]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Here’s a quick pizza joke for you:\n\n**Why did the pizza go to therapy?**\nBecause it had too many *toppings* from its past!\n\n(And now it’s *well-balanced*… just like a good pie should be. 🍕😆)', 'explanation': '**Joke:** Why don’t skeletons fight each other? *Because they don’t have the guts.*\n\n**Explanation:** The humor comes from the pun on "guts," meaning both courage and internal organs, which skeletons lack. Classic wordplay! (30 words)'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f156a5e-f57e-6a84-8002-624fd576e105'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-05-23T12:50:11.030441+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f156a5e-ebb9-649a-8001-f58dc179562d'}}, tasks=(), interrupts=())

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Here’s a quick pizza joke for you:\n\n**Why did the pizza go to therapy?**\nBecause it had too many *toppings* from its past!\n\n(And now it’s *well-balanced*… just like a good pie should be. 🍕😆)', 'explanation': '**Joke:** Why don’t skeletons fight each other? *Because they don’t have the guts.*\n\n**Explanation:** The humor comes from the pun on "guts," meaning both courage and internal organs, which skeletons lack. Classic wordplay! (30 words)'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f156a5e-f57e-6a84-8002-624fd576e105'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-05-23T12:50:11.030441+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f156a5e-ebb9-649a-8001-f58dc179562d'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Here’s a quick pizza joke for you:\n\n**Why did the pizz